## CCTV Weapon Threat Data Preparation Pipeline

This notebook prepares raw CCTV videos (e.g., SCVD, UCF Crime, generic action datasets) into a structured format suitable for:
- **YOLO-based weapon detection** (pistol, rifle, knife)
- **DeepSORT / ByteTrack tracking** and downstream temporal reasoning ("threat persistence", weapon drawing actions).

Each section is organized as a stage in the data engineering pipeline with clear configuration points for your datasets and environment.

### 1. Environment Setup

Install and import core dependencies. Run the `pip` cell only once per environment.

In [ ]:
# If running in a fresh environment, uncomment and execute this cell.
%pip install opencv-python numpy pandas tqdm kagglehub


In [ ]:
import os
import cv2
import math
import json
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from tqdm import tqdm
except ImportError:  # tqdm is optional
    def tqdm(x, **kwargs):
        return x

print("OpenCV version:", cv2.__version__)


### 2b. (Optional) Download videos from Kaggle into RAW_VIDEO_ROOT

Use **kagglehub** to download one or more video datasets from Kaggle into `RAW_VIDEO_ROOT`.  
- **`KAGGLE_DATASETS`**: list of `{"handle": "owner/dataset-name", "subfolder": "Name"}`. Each dataset is downloaded into `RAW_VIDEO_ROOT/<subfolder>/`, matching the format expected by the pipeline (one subfolder per dataset, videos inside).  
- Handle = from dataset URL: `https://www.kaggle.com/datasets/OWNER/DATASET_NAME` → `"owner/dataset-name"`.  
- Authenticate once via [Kaggle API token](https://www.kaggle.com/settings) (e.g. `kagglehub.login()` or `KAGGLE_API_TOKEN` env var).

In [ ]:
import shutil
import kagglehub
from pathlib import Path

# List of Kaggle datasets: each goes to RAW_VIDEO_ROOT/<subfolder>/ for consistent pipeline format.
# Handle = "owner/dataset-name" from https://www.kaggle.com/datasets/OWNER/DATASET_NAME
KAGGLE_DATASETS = [
    # {"handle": "webadvisor/real-time-anomaly-detection-in-cctv-surveillance", "subfolder": "webadvisor"},
    {"handle": "toluwaniaremu/smartcity-cctv-violence-detection-dataset-scvd", "subfolder": "toluwaniaremu"},
    {"handle": "jonathannield/cctv-action-recognition-dataset", "subfolder": "jonathannield"},
    # Add more: {"handle": "owner/name", "subfolder": "FolderName"},
]

# Same root as section 2 so format is consistent (run before or after section 2)
raw_root = Path("./raw_videos")
raw_root.mkdir(parents=True, exist_ok=True)

for item in KAGGLE_DATASETS:
    handle, subfolder = item["handle"], item["subfolder"]
    dest_dir = raw_root / subfolder
    dest_dir.mkdir(parents=True, exist_ok=True)
    # dataset_download() has no output_dir; it returns the cache path, then we copy into RAW_VIDEO_ROOT
    cache_path = Path(kagglehub.dataset_download(handle))
    for src in cache_path.rglob("*"):
        if src.is_file():
            rel = src.relative_to(cache_path)
            dst = dest_dir / rel
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
    print(f"Downloaded {handle} -> {dest_dir}")

### 2. Configuration: Paths and Class Mapping

Configure where your raw videos live and where processed frames, YOLO labels, and metadata should be written.

- **`RAW_VIDEO_ROOT`**: Root folder containing sub-folders per dataset (e.g., SCVD, UCF_Crime).
- **`OUTPUT_ROOT`**: Root folder for extracted frames, YOLO label files, and `metadata.csv`.
- **`CLASS_NAME_TO_ID`**: Mapping from semantic class names to YOLO class IDs. Adjust according to your detection head.

In [ ]:
# Root directory where your raw CCTV videos are stored.
# Example structure:
# RAW_VIDEO_ROOT/
#   SCVD/*.mp4
#   UCF_Crime/*.mp4
#   ActionDataset/*.avi
RAW_VIDEO_ROOT = Path("./raw_videos")  # TODO: point this to your actual raw video root

# Output directory for frames, YOLO labels, and metadata.
OUTPUT_ROOT = Path("./prepared_data")
FRAMES_DIR = OUTPUT_ROOT / "frames"          # frames/<video_id>/*.jpg
LABELS_DIR = OUTPUT_ROOT / "labels"          # labels/<video_id>/*.txt
METADATA_PATH = OUTPUT_ROOT / "metadata.csv"  # global metadata file

# Ensure base output directories exist.
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FRAMES_DIR.mkdir(parents=True, exist_ok=True)
LABELS_DIR.mkdir(parents=True, exist_ok=True)

# YOLO class mapping for weapons (extend as needed)
CLASS_NAME_TO_ID = {
    "pistol": 0,
    "rifle": 1,
    "knife": 2,
}

CLASS_ID_TO_NAME = {v: k for k, v in CLASS_NAME_TO_ID.items()}


### 3. Utility Functions

Core helpers used throughout the pipeline:

- **Video ID extraction** (stable grouping key per video).
- **Directory helpers** for per-video frame and label folders.
- **Time conversion** from frame indices to timestamps.
- **ROI representation** for mimicking high-angle CCTV views.

In [ ]:
def get_video_id(video_path: Path) -> str:
    """Return a stable video identifier from a full path.

    By default, this is the stem (filename without extension).
    If different datasets share names, you can customize this
    to include parent folder names.
    """
    return video_path.stem


def get_video_frame_dir(video_id: str) -> Path:
    """Directory where frames for a given video ID will be stored."""
    d = FRAMES_DIR / video_id
    d.mkdir(parents=True, exist_ok=True)
    return d


def get_video_label_dir(video_id: str) -> Path:
    """Directory where YOLO TXT labels for a given video ID will be stored."""
    d = LABELS_DIR / video_id
    d.mkdir(parents=True, exist_ok=True)
    return d


def frame_idx_to_timestamp(frame_idx: int, fps: float) -> float:
    """Convert frame index to timestamp in seconds."""
    if fps <= 0.0:
        return 0.0
    return frame_idx / fps


def normalize_roi(roi, frame_width: int, frame_height: int):
    """Normalize an ROI defined in pixel coordinates to [0, 1].

    ROI format (pixel): (x_min, y_min, x_max, y_max).
    Returns (x_min_n, y_min_n, x_max_n, y_max_n).
    """
    if roi is None:
        return None
    x_min, y_min, x_max, y_max = roi
    return (
        x_min / frame_width,
        y_min / frame_height,
        x_max / frame_width,
        y_max / frame_height,
    )


### 4. Adaptive Frame Sampling & ROI Processing

This section implements **adaptive frame sampling** and **Region of Interest (ROI)** cropping:

- Frames are sampled at a configurable rate (e.g., every 5th frame) to retain temporal context while reducing redundancy.
- A rectangular ROI can be specified in pixel coordinates to mimic high-angle CCTV views that focus on ground-level human-weapon interactions.
- Extracted frames are stored under `frames/<video_id>/` to preserve temporal consistency for trackers like DeepSORT/ByteTrack.

In [ ]:
def extract_frames_from_video(
    video_path: Path,
    sampling_rate: int = 5,
    roi_pixels=None,
    resize_to=None,
    action_sequences=None,
):
    """Extract frames from a video with adaptive sampling and optional ROI.

    Parameters
    ----------
    video_path : Path
        Path to a .mp4 or .avi video file.
    sampling_rate : int, optional
        Sample every N-th frame (e.g., 5 => keep frames 0, 5, 10, ...).
    roi_pixels : tuple or None, optional
        Optional ROI in pixel coords: (x_min, y_min, x_max, y_max).
        If provided, frames are cropped to this ROI before saving.
    resize_to : tuple or None, optional
        Optional resize (width, height) after cropping. If None, original
        resolution (or cropped resolution) is preserved.
    action_sequences : list of dict or None, optional
        Optional list of action sequence annotations for this video, each
        with keys: {"start_sec", "end_sec", "label"} where label could be
        "Active_Threat" or similar. Used for metadata tagging.

    Returns
    -------
    list of dict
        One record per extracted frame with keys such as:
        video_id, frame_idx, frame_path, timestamp_sec, is_active_threat, action_label.
    """
    video_path = Path(video_path)
    assert video_path.exists(), f"Video not found: {video_path}"

    video_id = get_video_id(video_path)
    frame_dir = get_video_frame_dir(video_id)

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Failed to open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS) or 0.0
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)

    records = []
    if action_sequences is None:
        action_sequences = []

    def get_action_label(ts: float):
        """Return action label for a given timestamp, if any."""
        for seq in action_sequences:
            if seq["start_sec"] <= ts <= seq["end_sec"]:
                return seq.get("label", "Active_Threat")
        return None

    frame_idx = 0
    extracted_idx = 0
    pbar = tqdm(total=frame_count, desc=f"Extracting {video_id}") if frame_count > 0 else None

    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break

            if frame_idx % sampling_rate == 0:
                h, w = frame.shape[:2]

                # Apply ROI crop (mimic high-angle camera focusing on ground plane / walkway).
                if roi_pixels is not None:
                    x_min, y_min, x_max, y_max = roi_pixels
                    x_min = max(0, int(x_min))
                    y_min = max(0, int(y_min))
                    x_max = min(w, int(x_max))
                    y_max = min(h, int(y_max))
                    frame = frame[y_min:y_max, x_min:x_max]
                    h, w = frame.shape[:2]

                # Optional resize for network input resolution (e.g., 1280x720, 640x640 square).
                if resize_to is not None:
                    frame = cv2.resize(frame, resize_to, interpolation=cv2.INTER_AREA)
                    h, w = frame.shape[:2]

                timestamp_sec = frame_idx_to_timestamp(frame_idx, fps)
                action_label = get_action_label(timestamp_sec)
                is_active_threat = bool(action_label is not None and action_label.lower() == "active_threat")

                frame_filename = f"{video_id}_frame_{frame_idx:06d}.jpg"
                frame_path = frame_dir / frame_filename

                # Save frame in BGR (OpenCV default).
                cv2.imwrite(str(frame_path), frame)

                record = {
                    "video_id": video_id,
                    "frame_idx": frame_idx,
                    "frame_filename": frame_filename,
                    "frame_path": str(frame_path.resolve()),
                    "timestamp_sec": float(timestamp_sec),
                    "width": int(w),
                    "height": int(h),
                    "roi_pixels": roi_pixels,
                    "roi_normalized": normalize_roi(roi_pixels, w, h) if roi_pixels is not None else None,
                    "is_active_threat": is_active_threat,
                    "action_label": action_label,
                }
                records.append(record)
                extracted_idx += 1

            frame_idx += 1
            if pbar is not None:
                pbar.update(1)

    finally:
        if pbar is not None:
            pbar.close()
        cap.release()

    print(f"Video {video_id}: extracted {extracted_idx} frames (sampling_rate={sampling_rate}).")
    return records


### 5. YOLO Label Formatting

This section defines utilities to convert bounding box annotations into **YOLO TXT format**:

`class_id x_center y_center width height` (all normalized to \[0, 1\]).

You can plug in manual annotations or pre-existing labels from SCVD/UCF-style datasets
by mapping them into the expected schema.

Expected internal bounding box format (per object per frame):
- `class_name` or `class_id` (weapons: pistol, rifle, knife)
- `x_min`, `y_min`, `x_max`, `y_max` in **pixel coordinates** relative to the (cropped) frame.

In [ ]:
def bbox_pixels_to_yolo(x_min, y_min, x_max, y_max, img_w, img_h):
    """Convert pixel bbox to YOLO normalized (x_center, y_center, w, h).

    All outputs are in the range [0, 1].
    """
    # Clamp to image bounds
    x_min = max(0, min(x_min, img_w - 1))
    y_min = max(0, min(y_min, img_h - 1))
    x_max = max(0, min(x_max, img_w - 1))
    y_max = max(0, min(y_max, img_h - 1))

    w = x_max - x_min
    h = y_max - y_min
    x_center = x_min + w / 2.0
    y_center = y_min + h / 2.0

    return (
        x_center / img_w,
        y_center / img_h,
        w / img_w,
        h / img_h,
    )


def format_yolo_line(class_id: int, x_center: float, y_center: float, w: float, h: float) -> str:
    """Return a single-line YOLO annotation string."""
    return f"{class_id} {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}"


def write_yolo_labels_for_frame(
    video_id: str,
    frame_record: dict,
    objects: list,
):
    """Write a YOLO TXT label file for a single frame.

    Parameters
    ----------
    video_id : str
        Video identifier.
    frame_record : dict
        Metadata record for this frame (output of `extract_frames_from_video`).
        Must contain `frame_idx`, `width`, `height`, and `frame_filename`.
    objects : list of dict
        Each dict represents one object annotation with keys:
        - `class_name` or `class_id`
        - `x_min`, `y_min`, `x_max`, `y_max` (pixel coordinates)

    Returns
    -------
    Path
        Path to the created YOLO label file.
    """
    label_dir = get_video_label_dir(video_id)
    frame_idx = frame_record["frame_idx"]
    img_w = frame_record["width"]
    img_h = frame_record["height"]

    label_filename = f"{video_id}_frame_{frame_idx:06d}.txt"
    label_path = label_dir / label_filename

    lines = []
    for obj in objects:
        if "class_id" in obj:
            class_id = int(obj["class_id"])
        else:
            class_name = obj["class_name"].lower()
            if class_name not in CLASS_NAME_TO_ID:
                # Skip unknown classes by default; you can also raise.
                continue
            class_id = CLASS_NAME_TO_ID[class_name]

        x_min = float(obj["x_min"])
        y_min = float(obj["y_min"])
        x_max = float(obj["x_max"])
        y_max = float(obj["y_max"])

        x_c, y_c, w, h = bbox_pixels_to_yolo(x_min, y_min, x_max, y_max, img_w, img_h)
        lines.append(format_yolo_line(class_id, x_c, y_c, w, h))

    # Write out the YOLO TXT file (one frame per file).
    with open(label_path, "w", encoding="utf-8") as f:
        for line in lines:
            f.write(line + "\n")

    return label_path


### 6. Action Sequence Tagging (Active Threats)

To train models that understand **threat persistence** and weapon drawing actions (not just single frames),
we define **action sequences** per video.

Each sequence specifies a time window during which an "Active Threat" (or other label) is present.

Schema for action sequences per video:
- `video_id`: identifier matching the frame extraction step.
- `start_sec`, `end_sec`: time window in seconds.
- `label`: semantic tag (e.g., `Active_Threat`, `Weapon_Visible`, `Weapon_Drawn`).

The `extract_frames_from_video` function consumes these annotations to set
`is_active_threat` and `action_label` per frame in the metadata.

In [ ]:
# Template structure for action sequences. In practice, you might load this
# from a CSV/JSON that encodes temporal annotations per video.

# Example (to be replaced with your real annotations):
# ACTION_SEQUENCES = {
#     "video_001": [
#         {"start_sec": 10.0, "end_sec": 25.0, "label": "Active_Threat"},
#         {"start_sec": 40.0, "end_sec": 50.0, "label": "Weapon_Visible"},
#     ],
# }
ACTION_SEQUENCES = {}


def get_action_sequences_for_video(video_id: str):
    """Return list of action sequence dicts for a given video_id."""
    return ACTION_SEQUENCES.get(video_id, [])


### 7. Batch Processing of Video Datasets

This step walks over your raw CCTV datasets (`RAW_VIDEO_ROOT`), extracts frames
using adaptive sampling and ROI processing, and accumulates metadata for each frame.

The temporal grouping by `video_id` (and per-video frame folders) is **critical**
for integrating with **DeepSORT/ByteTrack** and any sequence-level models.

In [ ]:
def discover_videos(root: Path, exts=(".mp4", ".avi", ".mov", ".mkv")):
    """Yield all video file paths under a root directory matching allowed extensions."""
    root = Path(root)
    for path in root.rglob("*"):
        if path.suffix.lower() in exts:
            yield path


def process_all_videos(
    raw_root: Path,
    sampling_rate: int = 5,
    roi_pixels=None,
    resize_to=None,
):
    """Process all videos under `raw_root` and accumulate metadata records.

    This function does **not** create labels by itself; it focuses on frame
    extraction and temporal metadata. Label creation is handled separately
    once bounding boxes are available.
    """
    all_records = []
    videos = list(discover_videos(raw_root))
    print(f"Discovered {len(videos)} videos under {raw_root}.")

    for video_path in videos:
        video_id = get_video_id(video_path)
        action_seqs = get_action_sequences_for_video(video_id)

        records = extract_frames_from_video(
            video_path=video_path,
            sampling_rate=sampling_rate,
            roi_pixels=roi_pixels,
            resize_to=resize_to,
            action_sequences=action_seqs,
        )
        all_records.extend(records)

    return all_records


### 8. Metadata Logging (`metadata.csv`)

We now aggregate all frame-level metadata into a single `metadata.csv` file for use by
your **Database Module** or downstream data loaders.

Each row can include:
- `video_id`, `frame_idx`, `frame_path`, `frame_filename`
- `timestamp_sec`, `width`, `height`
- `roi_pixels`, `roi_normalized`
- `is_active_threat`, `action_label`
- optional paths to YOLO label files (if labels have been created).

In [ ]:
def attach_label_paths_to_metadata(records: list):
    """Optionally attach YOLO label file paths to existing frame records.

    This assumes label files are named consistently with frames, e.g.:
    - frame:  <video_id>_frame_000010.jpg
    - label:  <video_id>_frame_000010.txt
    """
    for rec in records:
        video_id = rec["video_id"]
        frame_idx = rec["frame_idx"]
        label_dir = get_video_label_dir(video_id)
        label_filename = f"{video_id}_frame_{frame_idx:06d}.txt"
        label_path = label_dir / label_filename
        rec["yolo_label_path"] = str(label_path.resolve()) if label_path.exists() else None

    return records


def save_metadata(records: list, csv_path: Path = METADATA_PATH):
    """Save frame-level metadata records to `metadata.csv`."""
    if not records:
        print("No records to save.")
        return

    df = pd.DataFrame(records)
    # Ensure deterministic column order for reproducibility.
    preferred_cols = [
        "video_id",
        "frame_idx",
        "frame_filename",
        "frame_path",
        "timestamp_sec",
        "width",
        "height",
        "roi_pixels",
        "roi_normalized",
        "is_active_threat",
        "action_label",
        "yolo_label_path",
    ]
    cols = [c for c in preferred_cols if c in df.columns] + [
        c for c in df.columns if c not in preferred_cols
    ]
    df = df[cols]

    csv_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(csv_path, index=False)
    print(f"Saved metadata with {len(df)} rows to {csv_path}.")


### 9. End-to-End Example Usage

This cell demonstrates how to run the full pipeline for your CCTV datasets:

1. Discover all videos under `RAW_VIDEO_ROOT`.
2. Extract frames with adaptive sampling and ROI cropping.
3. (Optionally) create YOLO labels using your annotation source.
4. Attach label paths and save a global `metadata.csv`.

Adjust the configuration parameters (paths, ROI, sampling rate, resize) as needed.

In [ ]:
# --- Configuration example for a high-angle CCTV ROI ---
# Define an ROI in pixel coordinates after inspecting a representative frame.
# For example, crop the lower 60% of the frame to focus on human-weapon interactions.
EXAMPLE_ROI_PIXELS = None  # e.g., (0, int(0.4 * H), W, H) after you know W, H

# Sampling every 5th frame keeps temporal signal for trackers while
# reducing redundancy and disk usage.
SAMPLING_RATE = 5

# Optional resize for network training resolution (width, height), e.g., (1280, 720) or (640, 640).
RESIZE_TO = None


if __name__ == "__main__":
    # 1) Extract frames and basic metadata for all videos.
    records = process_all_videos(
        raw_root=RAW_VIDEO_ROOT,
        sampling_rate=SAMPLING_RATE,
        roi_pixels=EXAMPLE_ROI_PIXELS,
        resize_to=RESIZE_TO,
    )

    # 2) At this point, you would typically integrate your annotation source
    # (manual labels, SCVD/UCF bounding boxes, etc.) and call
    # `write_yolo_labels_for_frame` for each annotated frame.
    #
    # Pseudo-code (to be replaced with your actual loader):
    # for rec in records:
    #     video_id = rec["video_id"]
    #     frame_idx = rec["frame_idx"]
    #     objects = load_objects_for_frame(video_id, frame_idx)
    #     if objects:
    #         write_yolo_labels_for_frame(video_id, rec, objects)

    # 3) Attach label paths (if any labels exist) and save metadata CSV.
    records = attach_label_paths_to_metadata(records)
    save_metadata(records, csv_path=METADATA_PATH)

    print("Pipeline completed.")
